# MoCo-Tiny

MoCo frames contrastive learning as a **dictionary look-up**. Each image gives two augmented views: a **query** $x_q$ and a **key** $x_k$. The query must match its own key (the positive) against a large set of negative keys.

## Two encoders

MoCo uses **two** networks with the same architecture:

- the **query encoder** $f_q$ (+ projection head $g_q$), trained by gradient descent;
- the **key encoder** $f_k$ (+ projection head $g_k$), which receives **no gradient**.

The key encoder is a slowly-moving copy of the query encoder, updated by an **exponential moving average** (momentum update):

$$\theta_k \leftarrow m\,\theta_k + (1-m)\,\theta_q, \qquad m = 0.999$$

A high momentum keeps the key encoder almost frozen, so the keys stored over many steps stay
**consistent** with each other.

## Queue of negatives

Instead of taking negatives from the current batch (like SimCLR), MoCo keeps a **FIFO queue** of the last `QUEUE_SIZE` keys. This **decouples the number of negatives from the batch size**: with a
batch of 128 we still compare each query against thousands of negatives.

## InfoNCE loss

For a query $q = g_q(f_q(x_q))$, its positive key $k^{+} = g_k(f_k(x_k))$ and the queue
$\{k^{-}\}$, the loss is

$$
\mathcal{L} = - \log
\frac{\exp(q^\top k^{+} / \tau)}
{\exp(q^\top k^{+} / \tau) + \displaystyle\sum_{k^{-}} \exp(q^\top k^{-} / \tau)}
$$

All vectors are $\ell_2$-normalized, so the dot products are cosine similarities. This is a softmax cross-entropy where the positive is always class $0$.

## Architecture of $f$ and $g$

**Encoder $f$ : a ViT-Tiny backbone**. The image is split into $4\times4$ patches, linearly projectedto $d_{\text{model}}=128$, run through a 6-layer pre-norm Transformer, and the representation $h$ is the **global average pooling** over the patch tokens (no CLS token), matching the GAP pooling of the original MoCo ResNet.

**Projection head $g$ : a 2-layer MLP** ($128 \to 256 \to 128$ with BatchNorm + ReLU). The contrastive loss is computed on $z = g(h)$. **$g$ is discarded after pre-training**; downstream
evaluation uses only $h$ from the query encoder $f_q$.

![MoCo](figures/moco_architecture.png)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import math
import os

DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "DATA"))
CKPT_DIR = "./checkpoints/7_MoCo"
os.makedirs(CKPT_DIR, exist_ok=True)

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")

---
# 0. Dataset

**STL-10 unlabeled** (100k images) for self-supervised pre-training; **CIFAR-10** (50k train / 10k test) for downstream evaluation (linear probe + full fine-tune).

In [ ]:
BATCH_SIZE = 128
IMG_SIZE   = 32
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2470, 0.2435, 0.2616)


class TwoViews:
    def __init__(self, transform):
        self.transform = transform
    def __call__(self, x):
        return self.transform(x), self.transform(x)


color_jitter = transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)
ssl_aug = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([color_jitter], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
train_probe_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

pretrain_set = torchvision.datasets.STL10(root=DATA_DIR, split="unlabeled", download=True,
                                           transform=TwoViews(ssl_aug))
probe_train  = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=train_probe_tf)
probe_test   = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=test_tf)

pretrain_loader     = DataLoader(pretrain_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, drop_last=True)
probe_train_loader  = DataLoader(probe_train,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
probe_test_loader   = DataLoader(probe_test,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

sample_img, _ = probe_train[0]
CHANNELS, H, W = sample_img.shape
NUM_CLASSES    = len(probe_train.classes)
print(f"pretrain {len(pretrain_set)} | probe train {len(probe_train)} | probe test {len(probe_test)}")

### Data augmentations

MoCo uses the **same contrastive augmentations as SimCLR**: each image is passed twice through
`ssl_aug` to produce a query view and a key view. We use the four operators of the winning
**crop + color distortion** family.

| Operator | In the code |
|---|---|
| crop and resize | `RandomResizedCrop(32, scale=(0.2, 1.0))` |
| horizontal flip | `RandomHorizontalFlip(p=0.5)` |
| color jitter | `RandomApply([ColorJitter(0.4,0.4,0.4,0.1)], p=0.8)` |
| color drop (grayscale) | `RandomGrayscale(p=0.2)` |

The downstream probe / fine-tune use only the mild standard recipe (`RandomCrop(32, padding=4)` + flip).

In [ ]:
def unnormalize(img):
    m = torch.tensor(MEAN).view(3, 1, 1)
    s = torch.tensor(STD).view(3, 1, 1)
    return (img * s + m).clamp(0, 1)

display_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

from PIL import Image
indices = torch.randint(0, len(pretrain_set), (10,))
fig, axes = plt.subplots(3, 10, figsize=(16, 5))
for i, idx in enumerate(indices):
    raw = pretrain_set.data[idx]
    orig = display_tf(Image.fromarray(raw.transpose(1, 2, 0)))
    (v1, v2), _ = pretrain_set[idx]
    for r, v in [(0, orig), (1, v1), (2, v2)]:
        axes[r, i].imshow(unnormalize(v).permute(1, 2, 0).numpy())
        axes[r, i].set_xticks([]); axes[r, i].set_yticks([])
axes[0, 0].set_ylabel("original", fontsize=11)
axes[1, 0].set_ylabel("query",    fontsize=11)
axes[2, 0].set_ylabel("key",      fontsize=11)
plt.suptitle("MoCo augmentations: query and key views of the same image", fontsize=13)
plt.tight_layout(); plt.show()

---
# 1. Model & hyperparameters

Shared **ViT-Tiny** encoder $f$ (global average pooling, 128-d feature $h$) followed by a 2-layer
MLP projection head $g$. MoCo adds a momentum copy of $(f, g)$ and a queue of negatives. The
detailed description is at the top of the notebook.

In [ ]:
PATCH_SIZE  = 4
GRID_SIZE   = IMG_SIZE // PATCH_SIZE
N_PATCHES   = GRID_SIZE ** 2
PATCH_DIM   = PATCH_SIZE ** 2 * CHANNELS

D_MODEL    = 128
NUM_HEADS  = 4
NUM_LAYERS = 6
D_FF       = 4 * D_MODEL
DROPOUT    = 0.1

PRETRAIN_EPOCHS = 20
PROBE_EPOCHS    = 60
FT_EPOCHS       = 60

PROJ_DIM    = 128
QUEUE_SIZE  = 4096
MOMENTUM    = 0.999
TEMPERATURE = 0.2

In [ ]:
def patchify(images, patch_size):
    B, C, H, W = images.shape
    P = patch_size
    x = images.reshape(B, C, H // P, P, W // P, P)
    x = x.permute(0, 2, 4, 3, 5, 1)
    x = x.reshape(B, (H // P) * (W // P), P * P * C)
    return x

In [ ]:
class ViTEncoder(nn.Module):
    
    def __init__(self, img_size=IMG_SIZE, patch_size=PATCH_SIZE, channels=CHANNELS, d_model=D_MODEL, num_heads=NUM_HEADS, num_layers=NUM_LAYERS, d_ff=D_FF, dropout=DROPOUT):
        
        super().__init__()
        n_patches = (img_size // patch_size) ** 2
        patch_dim = patch_size ** 2 * channels
        self.patch_size = patch_size

        self.projection    = nn.Linear(patch_dim, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(1, n_patches, d_model) * 0.02)
        self.dropout       = nn.Dropout(dropout)

        block = nn.TransformerEncoderLayer( d_model=d_model, nhead=num_heads, dim_feedforward=d_ff,dropout=dropout, activation="gelu", batch_first=True, norm_first=True,) 
        self.transformer = nn.TransformerEncoder(block, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, images):
        x = patchify(images, self.patch_size)
        x = self.projection(x) + self.pos_embedding
        x = self.dropout(x)
        x = self.transformer(x)
        x = self.norm(x)
        return x.mean(dim=1)   # x (B, d_model)

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, in_dim=D_MODEL, hidden=D_MODEL*2, out_dim=PROJ_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, h): return self.net(h)


class MoCo(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder_q = ViTEncoder()
        self.proj_q = ProjectionHead()
        self.encoder_k = ViTEncoder()
        self.proj_k = ProjectionHead()
        for pq, pk in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            pk.data.copy_(pq.data)
            pk.requires_grad = False
        for pq, pk in zip(self.proj_q.parameters(), self.proj_k.parameters()):
            pk.data.copy_(pq.data)
            pk.requires_grad = False
        self.register_buffer("queue", F.normalize(torch.randn(PROJ_DIM, QUEUE_SIZE), dim=0))
        self.register_buffer("queue_ptr", torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def _update_key(self):
        m = MOMENTUM
        for pq, pk in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            pk.data.mul_(m).add_(pq.data, alpha=1.0 - m)
        for pq, pk in zip(self.proj_q.parameters(), self.proj_k.parameters()):
            pk.data.mul_(m).add_(pq.data, alpha=1.0 - m)

    @torch.no_grad()
    def _enqueue(self, keys):
        B = keys.shape[0]; ptr = int(self.queue_ptr.item())
        if ptr + B <= QUEUE_SIZE:
            self.queue[:, ptr:ptr+B] = keys.T
        else:
            tail = QUEUE_SIZE - ptr
            self.queue[:, ptr:] = keys[:tail].T
            self.queue[:, :B - tail] = keys[tail:].T
        self.queue_ptr[0] = (ptr + B) % QUEUE_SIZE

    def forward(self, xq, xk):
        q = F.normalize(self.proj_q(self.encoder_q(xq)), dim=-1)
        with torch.no_grad():
            self._update_key()
            k = F.normalize(self.proj_k(self.encoder_k(xk)), dim=-1)
        l_pos = (q * k).sum(dim=-1, keepdim=True)
        l_neg = q @ self.queue.clone().detach()
        logits = torch.cat([l_pos, l_neg], dim=1) / TEMPERATURE
        labels = torch.zeros(logits.shape[0], dtype=torch.long, device=logits.device)
        loss = F.cross_entropy(logits, labels)
        self._enqueue(k)
        return loss


moco = MoCo().to(device)
trainable = list(moco.encoder_q.parameters()) + list(moco.proj_q.parameters())
print(f"trainable params : {sum(p.numel() for p in trainable):,}")

---
# 2. Pre-training on STL-10

Train the query encoder $f_q$ and head $g_q$ with the InfoNCE loss for a fixed budget of 20 epochs.
The key encoder follows by momentum; the queue is filled as training proceeds.

In [ ]:
optimizer = torch.optim.AdamW(trainable, lr=3e-4, weight_decay=1e-4)
pretrain_losses = []
for epoch in range(1, PRETRAIN_EPOCHS + 1):
    moco.train()
    total, n = 0.0, 0
    for (v1, v2), _ in pretrain_loader:
        v1, v2 = v1.to(device), v2.to(device)
        loss = moco(v1, v2)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * v1.size(0); n += v1.size(0)
    avg = total / n
    pretrain_losses.append(avg)
    print(f"pretrain {epoch:2d}/{PRETRAIN_EPOCHS} | loss {avg:.4f}")
    torch.save({"epoch": epoch, "model": moco.state_dict(), "losses": pretrain_losses},
               os.path.join(CKPT_DIR, "moco_pretrain.pt"))

## Curves

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(range(1, len(pretrain_losses) + 1), pretrain_losses, marker="o")
plt.title("MoCo pre-training loss"); plt.xlabel("epoch"); plt.ylabel("InfoNCE")
plt.grid(True, alpha=0.3); plt.show()

---
# 3. Encoder for probe / fine-tune

Reload the best pre-trained model and expose its **query encoder** $f_q$. Downstream tasks use the
encoder output $h$ -- the projection head $g$ and the key encoder are **discarded**. We also define
the slope diagnostic and the train / eval helpers.

In [ ]:
def build_eval_encoder():
    m = MoCo().to(device)
    ckpt = torch.load(os.path.join(CKPT_DIR, "moco_pretrain.pt"), map_location=device)
    m.load_state_dict(ckpt["model"])
    return m.encoder_q

In [ ]:
def slope_per_epoch(values, window=5):
    if len(values) < 2:
        return float("inf")
    w = min(window, len(values))
    y = np.array(values[-w:]); x = np.arange(w)
    return float(np.polyfit(x, y, 1)[0])


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += (logits.argmax(dim=1) == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)
        correct    += (logits.argmax(dim=1) == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total

---
# 4. Linear probe on CIFAR-10

Freeze the query encoder, train only a linear classifier head on the frozen feature $h$.

In [ ]:
class LinearProbe(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.head    = nn.Linear(D_MODEL, num_classes)
        for p in self.encoder.parameters():
            p.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            h = self.encoder(x)
        return self.head(h)

In [ ]:
probe = LinearProbe(build_eval_encoder(), NUM_CLASSES).to(device)
print(f"trainable : {sum(p.numel() for p in probe.head.parameters()):,}")

In [ ]:
PROBE_LR = 1e-3

optimizer = torch.optim.AdamW(probe.head.parameters(), lr=PROBE_LR, weight_decay=0.0)
criterion = nn.CrossEntropyLoss()

PATIENCE = 8
train_losses, test_losses = [], []
train_accs,   test_accs   = [], []
best_acc = 0.0

for epoch in range(1, PROBE_EPOCHS + 1):
    probe.train()
    probe.encoder.eval()
    tr_loss, tr_acc = train_one_epoch(probe, probe_train_loader, optimizer, criterion)
    te_loss, te_acc = evaluate(probe, probe_test_loader, criterion)
    train_losses.append(tr_loss); test_losses.append(te_loss)
    train_accs.append(tr_acc);   test_accs.append(te_acc)
    print(f"epoch {epoch:2d}/{PROBE_EPOCHS} | train {tr_loss:.4f} {tr_acc*100:.1f}% | test {te_loss:.4f} {te_acc*100:.1f}%")

    if te_acc > best_acc:
        best_acc = te_acc
        torch.save({"epoch": epoch, "model": probe.state_dict(),
                    "test_accs": test_accs, "train_accs": train_accs,
                    "test_losses": test_losses, "train_losses": train_losses},
                   os.path.join(CKPT_DIR, "moco_probe_best.pt"))

    best_epoch = test_accs.index(max(test_accs))
    if (len(test_accs) - 1 - best_epoch) >= PATIENCE:
        print(f"  early stop : converged (no improvement for {PATIENCE} epochs)")
        break

print()
print(f"best test acc : {max(test_accs)*100:.2f}%")
print(f"converged at  : epoch {test_accs.index(max(test_accs))+1}")
print(f"final slope   : {slope_per_epoch([a*100 for a in test_accs]):.3f} %/epoch")

## Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label="train"); ax1.plot(test_losses, label="test")
ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.legend(); ax1.set_title("Loss")
ax2.plot([a*100 for a in train_accs], label="train"); ax2.plot([a*100 for a in test_accs], label="test")
ax2.set_xlabel("epoch"); ax2.set_ylabel("acc (%)"); ax2.legend(); ax2.set_title("Accuracy")
plt.tight_layout(); plt.show()

## Results

Plot 10 random predictions from the test set.

In [ ]:
probe.eval()
classes = probe_test.classes
indices = torch.randint(0, len(probe_test), (10,))

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, idx in enumerate(indices):
    img, true_label = probe_test[idx]
    with torch.no_grad():
        logits = probe(img.unsqueeze(0).to(device))
        pred   = logits.argmax(dim=1).item()
    axes[i].imshow(unnormalize(img).permute(1, 2, 0).cpu().numpy())
    color = "green" if pred == true_label else "red"
    axes[i].set_title(f"pred: {classes[pred]}\ntrue: {classes[true_label]}",
                       fontsize=6, color=color)
    axes[i].axis("off")
plt.tight_layout(); plt.show()

---
# 5. Full fine-tune on CIFAR-10

Unfreeze the query encoder, train everything end-to-end with a small learning rate.

In [ ]:
class FullFineTune(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.head    = nn.Linear(D_MODEL, num_classes)

    def forward(self, x):
        h = self.encoder(x)
        return self.head(h)

In [ ]:
ft = FullFineTune(build_eval_encoder(), NUM_CLASSES).to(device)
print(f"trainable : {sum(p.numel() for p in ft.parameters()):,}")

In [ ]:
FT_LR = 1e-4

optimizer = torch.optim.AdamW(ft.parameters(), lr=FT_LR, weight_decay=0.05)
criterion = nn.CrossEntropyLoss()

PATIENCE = 8
train_losses, test_losses = [], []
train_accs,   test_accs   = [], []
best_acc = 0.0

for epoch in range(1, FT_EPOCHS + 1):
    ft.train()
    tr_loss, tr_acc = train_one_epoch(ft, probe_train_loader, optimizer, criterion)
    te_loss, te_acc = evaluate(ft, probe_test_loader, criterion)
    train_losses.append(tr_loss); test_losses.append(te_loss)
    train_accs.append(tr_acc);   test_accs.append(te_acc)
    print(f"epoch {epoch:2d}/{FT_EPOCHS} | train {tr_loss:.4f} {tr_acc*100:.1f}% | test {te_loss:.4f} {te_acc*100:.1f}%")

    if te_acc > best_acc:
        best_acc = te_acc
        torch.save({"epoch": epoch, "model": ft.state_dict(),
                    "test_accs": test_accs, "train_accs": train_accs,
                    "test_losses": test_losses, "train_losses": train_losses},
                   os.path.join(CKPT_DIR, "moco_ft_best.pt"))

    best_epoch = test_accs.index(max(test_accs))
    if (len(test_accs) - 1 - best_epoch) >= PATIENCE:
        print(f"  early stop : converged (no improvement for {PATIENCE} epochs)")
        break

print()
print(f"best test acc : {max(test_accs)*100:.2f}%")
print(f"converged at  : epoch {test_accs.index(max(test_accs))+1}")
print(f"final slope   : {slope_per_epoch([a*100 for a in test_accs]):.3f} %/epoch")

## Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label="train"); ax1.plot(test_losses, label="test")
ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.legend(); ax1.set_title("Loss")
ax2.plot([a*100 for a in train_accs], label="train"); ax2.plot([a*100 for a in test_accs], label="test")
ax2.set_xlabel("epoch"); ax2.set_ylabel("acc (%)"); ax2.legend(); ax2.set_title("Accuracy")
plt.tight_layout(); plt.show()

## Results

Plot 10 random predictions from the test set.

In [ ]:
ft.eval()
classes = probe_test.classes
indices = torch.randint(0, len(probe_test), (10,))

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, idx in enumerate(indices):
    img, true_label = probe_test[idx]
    with torch.no_grad():
        logits = ft(img.unsqueeze(0).to(device))
        pred   = logits.argmax(dim=1).item()
    axes[i].imshow(unnormalize(img).permute(1, 2, 0).cpu().numpy())
    color = "green" if pred == true_label else "red"
    axes[i].set_title(f"pred: {classes[pred]}\ntrue: {classes[true_label]}",
                       fontsize=6, color=color)
    axes[i].axis("off")
plt.tight_layout(); plt.show()